In [ ]:
# Cell 1：依赖、SI 单位宏变量与附件 1 参数
from dataclasses import dataclass, field
from typing import Optional, Sequence

import numpy as np

# -----------------------------------------------------------------------------
# 统一约定：求解器内部全部使用 SI 单位
# 电流密度 A/m^2，长度 m，温度 K，压力 Pa，浓度 mol/m^3，水含量 kg/m^3
# -----------------------------------------------------------------------------
F = 96485.0                         # C/mol
R = 8.314                           # J/(mol K)
T_REF = 298.15                      # K
P_REF = 101325.0                    # Pa
T_FREEZE = 273.15                   # K
E_TH = 1.48                         # V，热中性电压
E_OPEN_INITIAL = 0.95               # V，附件 1 给出的初始开路电压

M_H2 = 2.016e-3                     # kg/mol
M_O2 = 31.998e-3                    # kg/mol
M_N2 = 28.014e-3                    # kg/mol
M_WATER = 0.018                     # kg/mol
RHO_ICE = 920.0                     # kg/m^3
RHO_LIQUID = 990.0                  # kg/m^3
LATENT_CONDENSATION = 2.50e6        # J/kg
LATENT_FREEZING = 333600.0          # J/kg

AREA_CELL = 25.0e-4                 # m^2，25 cm^2
L_AGDL = 150.0e-6                   # m
L_ACL = 3.4e-6                      # m
L_PEM = 12.0e-6                     # m
L_CCL = 11.3e-6                     # m
L_CGDL = 150.0e-6                   # m
L_TOTAL = L_AGDL + L_ACL + L_PEM + L_CCL + L_CGDL
L_DIFF_CATHODE = L_CCL + L_CGDL

EPS_AGDL = 0.8
EPS_ACL = 0.3916
EPS_CCL = 0.4207
EPS_CGDL = 0.8
EPS_FLOOR = 1.0e-12

PERM_GDL = 6.2e-12                  # m^2
PERM_CL = 6.2e-13                   # m^2
CONTACT_ANGLE_GDL = 110.0           # deg
CONTACT_ANGLE_CL = 100.0            # deg

D_H2_REF = 1.10e-4                  # m^2/s
D_O2_REF = 2.20e-5                  # m^2/s
D_WATER_ANODE_REF = 8.69e-5         # m^2/s，aGDL/aCL 水蒸气
D_WATER_CATHODE_REF = 2.48e-5       # m^2/s，cCL/cGDL 水蒸气
DIFFUSIVITY_T_EXPONENT = 1.75
BRUGGEMAN_EXPONENT = 1.5

RHO_PEM = 2150.0                    # kg/m^3
EW_PEM = 1.0                        # kg/mol，1000 g/mol
LAMBDA_INITIAL = 3.0
CL_IONOMER_FRACTION = 0.3

ALPHA = 0.5
J0_REF = 0.01                       # A/m^2，题目建议的初始校准值
EA_ACTIVATION = 67000.0             # J/mol
R_CONTACT_ASR = 0.01e-4             # ohm m^2，0.01 ohm cm^2
BETA_ICE_ACTIVE_AREA = 3.5

H_CONVECTION = 40.0                 # W/(m^2 K)
T_INITIAL_DEFAULT = 253.15          # K，附件 1 默认 -20 degC
T_AMBIENT_DEFAULT = 253.15          # K
P_OPERATING = 101325.0              # Pa
Y_H2_IN = 1.0
Y_O2_IN = 0.233
Y_N2_IN = 0.767
Y_WATER_ANODE_IN = 0.0
Y_WATER_CATHODE_IN = 0.0

# 热物性统一写成 (rho [kg/m^3], cp [J/(kg K)], k [W/(m K)])。
# 数值均来自附件 1；气体密度是附件给出的参考密度，不替代理想气体浓度关系。
THERMAL_H2 = (0.089, 14283.0, 0.1672)
THERMAL_O2 = (1.43, 919.31, 0.0246)
THERMAL_N2 = (1.35, 1041.5, 0.0235)
THERMAL_VAPOR = (4.8e-3, 2000.0, 0.10)
THERMAL_LIQUID = (990.0, 4182.0, 0.60)
THERMAL_ICE = (920.0, 2050.0, 2.30)
THERMAL_GDL = (185.0, 545.0, 0.30)
THERMAL_CL = (970.0, 240.0, 0.27)
THERMAL_PEM = (2150.0, 1050.0, 0.24)

# 附件 1 的水传输/相变系数。成对数值按附件中的正向、反向顺序展开。
K_MEMBRANE_TO_VAPOR = 0.001
K_VAPOR_TO_MEMBRANE = 1.0
K_MEMBRANE_LIQUID = 0.5
K_MEMBRANE_ICE = 1.0
K_CONDENSATION = 1.0
K_EVAPORATION = 1.0
K_DESUBLIMATION = 1.0e-4

# 附件 1 未单列液水-冰速率；采用参考文献 [3] Jiao & Li (2009), Table 4。
K_FREEZE_LIQUID = 1.0             # 1/s
K_MELT_ICE = 1.0                  # 1/s

# 启动前已经吹扫：附件 1 第 55--56 行。
LIQUID_INITIAL = 0.0               # kg/m^3
ICE_FRACTION_INITIAL = 0.0           # -


CURRENT_A_CM2_TO_A_M2 = 1.0e4



def air_mass_to_mole_fraction(y_o2: float, y_n2: float) -> tuple[float, float]:
    """把附件 1 的干空气质量分数换算为摩尔分数。"""
    n_o2 = y_o2 / M_O2
    n_n2 = y_n2 / M_N2
    total = n_o2 + n_n2
    return n_o2 / total, n_n2 / total


X_O2_IN, X_N2_IN = air_mass_to_mole_fraction(Y_O2_IN, Y_N2_IN)


def ideal_gas_species_concentration(temperature, pressure, mole_fraction=1.0):
    """由局部 T、p 和摩尔分数计算气体摩尔浓度，单位 mol/m^3。"""
    temperature = np.asarray(temperature, dtype=float)
    pressure = np.asarray(pressure, dtype=float)
    mole_fraction = np.asarray(mole_fraction, dtype=float)
    if np.any(temperature <= 0.0) or np.any(pressure <= 0.0):
        raise ValueError("温度和绝对压力必须为正")
    if np.any((mole_fraction < 0.0) | (mole_fraction > 1.0)):
        raise ValueError("摩尔分数必须位于 [0, 1]")
    return mole_fraction * pressure / (R * temperature)


def C_H2_IN_DEFAULT(temperature, pressure=P_OPERATING):
    """纯氢入口浓度 c_H2(T, p)。"""
    return ideal_gas_species_concentration(temperature, pressure, mole_fraction=1.0)


def C_O2_IN_DEFAULT(temperature, pressure=P_OPERATING):
    """干空气入口氧浓度 c_O2(T, p)，质量分数先换算为摩尔分数。"""
    return ideal_gas_species_concentration(temperature, pressure, mole_fraction=X_O2_IN)

# 气体 Dirichlet 边界保存可调用函数，离散时使用边界处的瞬时 T、p 求值。
BOUNDARY_DEFAULTS = {
    "temperature_left": ("Robin", H_CONVECTION, T_AMBIENT_DEFAULT),
    "temperature_right": ("Robin", H_CONVECTION, T_AMBIENT_DEFAULT),
    "h2_left": ("Dirichlet", C_H2_IN_DEFAULT),
    "o2_right": ("Dirichlet", C_O2_IN_DEFAULT),
    "water_left": ("Dirichlet_mass_fraction", Y_WATER_ANODE_IN),
    "water_right": ("Dirichlet_mass_fraction", Y_WATER_CATHODE_IN),
}

RESIDUAL_TOLERANCE = 1.0e-6


: 

In [ ]:
# Cell 2：一维 PEMFC 冷启动物理模型
# 求解顺序：相态/孔隙率 -> 传输系数 -> 电压/反应 -> 质量方程 -> 热源/温度方程


def _cell_widths(dx, size):
    """把标量或逐单元宽度统一为长度为 size 的一维数组。"""
    widths = np.asarray(dx, dtype=float)
    if widths.ndim == 0:
        widths = np.full(size, float(widths))
    if widths.shape != (size,) or np.any(widths <= 0.0):
        raise ValueError("dx 必须为正标量或与单元数一致的一维数组")
    return widths


def _weighted_harmonic_face(coefficient, dx):
    """非均匀网格内部面的距离加权调和平均。"""
    coefficient = np.asarray(coefficient, dtype=float)
    if coefficient.ndim != 1 or coefficient.size < 2:
        raise ValueError("coefficient 必须是至少含两个单元的一维数组")
    widths = _cell_widths(dx, coefficient.size)
    left_distance = 0.5 * widths[:-1]
    right_distance = 0.5 * widths[1:]
    denominator = (
        left_distance / np.maximum(coefficient[:-1], EPS_FLOOR)
        + right_distance / np.maximum(coefficient[1:], EPS_FLOOR)
    )
    face = (left_distance + right_distance) / denominator
    blocked = (coefficient[:-1] <= EPS_FLOOR) | (coefficient[1:] <= EPS_FLOOR)
    return np.where(blocked, 0.0, face)


def _diffusive_face_flux(
    field,
    coefficient,
    dx,
    left_value=None,
    right_value=None,
    left_flux=0.0,
    right_flux=0.0,
):
    """构造 N+1 个面通量；正方向统一取 +x。"""
    field = np.asarray(field, dtype=float)
    coefficient = np.asarray(coefficient, dtype=float)
    if field.ndim != 1 or coefficient.shape != field.shape:
        raise ValueError("field 与 coefficient 必须为等长一维数组")
    widths = _cell_widths(dx, field.size)
    face_flux = np.zeros(field.size + 1, dtype=float)
    if field.size > 1:
        face_coefficient = _weighted_harmonic_face(coefficient, widths)
        center_distance = 0.5 * (widths[:-1] + widths[1:])
        face_flux[1:-1] = -face_coefficient * np.diff(field) / center_distance
    if left_value is None:
        face_flux[0] = left_flux
    else:
        face_flux[0] = -coefficient[0] * (field[0] - left_value) / (0.5 * widths[0])
    if right_value is None:
        face_flux[-1] = right_flux
    else:
        face_flux[-1] = -coefficient[-1] * (right_value - field[-1]) / (0.5 * widths[-1])
    return face_flux


def _face_divergence(face_flux, dx):
    """由 N+1 个面通量计算 N 个控制体的散度。"""
    face_flux = np.asarray(face_flux, dtype=float)
    if face_flux.ndim != 1 or face_flux.size < 2:
        raise ValueError("face_flux 必须是一维面通量数组")
    widths = _cell_widths(dx, face_flux.size - 1)
    return np.diff(face_flux) / widths


@dataclass
class GasConservation:
    """H2/O2：d(eps_g*c_k)/dt = -div(N_k) + S_k。"""

    d_h2_ref: float = D_H2_REF
    d_o2_ref: float = D_O2_REF
    t_ref: float = T_REF
    p_ref: float = P_REF
    temperature_exponent: float = DIFFUSIVITY_T_EXPONENT
    porosity_exponent: float = BRUGGEMAN_EXPONENT

    def effective_diffusivity(self, species, temperature, pressure, gas_porosity):
        species_key = species.lower()
        if species_key == "h2":
            d_ref = self.d_h2_ref
        elif species_key == "o2":
            d_ref = self.d_o2_ref
        else:
            raise ValueError("species 必须为 'h2' 或 'o2'")
        temperature = np.asarray(temperature, dtype=float)
        pressure = np.asarray(pressure, dtype=float)
        gas_porosity = np.asarray(gas_porosity, dtype=float)
        if np.any(temperature <= 0.0) or np.any(pressure <= 0.0):
            raise ValueError("温度和绝对压力必须为正")
        return (
            d_ref
            * (temperature / self.t_ref) ** self.temperature_exponent
            * (self.p_ref / pressure)
            * np.maximum(gas_porosity, EPS_FLOOR) ** self.porosity_exponent
        )

    @staticmethod
    def face_flux(
        concentration,
        diffusivity,
        dx,
        left_value=None,
        right_value=None,
        left_flux=0.0,
        right_flux=0.0,
    ):
        return _diffusive_face_flux(
            concentration,
            diffusivity,
            dx,
            left_value=left_value,
            right_value=right_value,
            left_flux=left_flux,
            right_flux=right_flux,
        )

    @staticmethod
    def rhs(concentration, gas_porosity, d_gas_porosity_dt, face_flux, source, dx):
        concentration = np.asarray(concentration, dtype=float)
        gas_porosity = np.asarray(gas_porosity, dtype=float)
        d_gas_porosity_dt = np.asarray(d_gas_porosity_dt, dtype=float)
        source = np.asarray(source, dtype=float)
        return (
            -_face_divergence(face_flux, dx)
            + source
            - concentration * d_gas_porosity_dt
        ) / np.maximum(gas_porosity, EPS_FLOOR)


@dataclass
class WaterConservation:
    """总水 PDE + 蒸汽/液水/冰分配 + 独立冰相控制方程。"""

    molar_mass: float = M_WATER
    rho_pem: float = RHO_PEM
    equivalent_weight: float = EW_PEM
    rho_liquid: float = RHO_LIQUID
    rho_ice: float = RHO_ICE
    k_freeze: float = K_FREEZE_LIQUID
    k_melt: float = K_MELT_ICE

    def membrane_water_content(self, water_mass):
        return (
            self.equivalent_weight
            * np.asarray(water_mass, dtype=float)
            / (self.rho_pem * self.molar_mass)
        )

    @staticmethod
    def electro_osmotic_drag(lambda_water):
        return 2.5 * np.asarray(lambda_water, dtype=float) / 22.0

    @staticmethod
    def membrane_diffusivity(temperature, lambda_water):
        temperature = np.asarray(temperature, dtype=float)
        lambda_water = np.asarray(lambda_water, dtype=float)
        if np.any(temperature <= 0.0):
            raise ValueError("绝对温度必须为正")
        polynomial = (
            2.563
            - 0.33 * lambda_water
            + 0.0264 * lambda_water**2
            - 0.000671 * lambda_water**3
        )
        diffusivity = (
            1.0e-10
            * np.exp(2416.0 * (1.0 / 303.15 - 1.0 / temperature))
            * polynomial
        )
        return np.maximum(diffusivity, 0.0)

    @staticmethod
    def saturation_pressure(temperature):
        """Buck 关系式；零上为水面、零下为冰面饱和蒸气压，单位 Pa。"""
        temperature = np.asarray(temperature, dtype=float)
        t_celsius = temperature - 273.15
        p_water = 611.21 * np.exp(
            (18.678 - t_celsius / 234.5) * t_celsius / (257.14 + t_celsius)
        )
        p_ice = 611.15 * np.exp(
            (23.036 - t_celsius / 333.7) * t_celsius / (279.82 + t_celsius)
        )
        return np.where(t_celsius >= 0.0, p_water, p_ice)

    def saturated_vapor_density(self, temperature):
        """单位气相体积内的饱和水蒸气质量浓度，kg/m^3。"""
        temperature = np.asarray(temperature, dtype=float)
        return self.molar_mass * self.saturation_pressure(temperature) / (R * temperature)

    def phase_state(self, total_water, ice_mass, temperature, initial_porosity):
        """瞬时饱和平衡：总水分为 mv、ml、mi，并同步扣除液水/冰占孔。"""
        total_water, ice_mass, temperature, initial_porosity = np.broadcast_arrays(
            np.maximum(np.asarray(total_water, dtype=float), 0.0),
            np.maximum(np.asarray(ice_mass, dtype=float), 0.0),
            np.asarray(temperature, dtype=float),
            np.asarray(initial_porosity, dtype=float),
        )
        ice_mass = np.minimum(ice_mass, total_water)
        mobile_water = total_water - ice_mass
        ice_fraction = ice_mass / self.rho_ice
        pore_without_liquid = np.maximum(initial_porosity - ice_fraction, EPS_FLOOR)
        vapor_saturation = self.saturated_vapor_density(temperature)
        denominator = np.maximum(1.0 - vapor_saturation / self.rho_liquid, EPS_FLOOR)
        liquid_mass = np.where(
            mobile_water > vapor_saturation * pore_without_liquid,
            (mobile_water - vapor_saturation * pore_without_liquid) / denominator,
            0.0,
        )
        liquid_mass = np.clip(liquid_mass, 0.0, mobile_water)
        vapor_mass = mobile_water - liquid_mass
        liquid_fraction = liquid_mass / self.rho_liquid
        gas_porosity = np.maximum(
            initial_porosity - liquid_fraction - ice_fraction,
            EPS_FLOOR,
        )
        return {
            "vapor_mass": vapor_mass,
            "liquid_mass": liquid_mass,
            "ice_mass": ice_mass,
            "gas_porosity": gas_porosity,
            "liquid_fraction": liquid_fraction,
            "ice_fraction": ice_fraction,
        }

    @staticmethod
    def vapor_effective_diffusivity(d_ref, temperature, pressure, gas_porosity):
        temperature = np.asarray(temperature, dtype=float)
        pressure = np.asarray(pressure, dtype=float)
        gas_porosity = np.asarray(gas_porosity, dtype=float)
        if np.any(temperature <= 0.0) or np.any(pressure <= 0.0):
            raise ValueError("温度和绝对压力必须为正")
        return (
            np.asarray(d_ref, dtype=float)
            * (temperature / T_REF) ** DIFFUSIVITY_T_EXPONENT
            * (P_REF / pressure)
            * np.maximum(gas_porosity, EPS_FLOOR) ** BRUGGEMAN_EXPONENT
        )

    @staticmethod
    def porous_face_flux(
        vapor_mass,
        diffusivity,
        dx,
        left_value=None,
        right_value=None,
        left_flux=0.0,
        right_flux=0.0,
    ):
        return _diffusive_face_flux(
            vapor_mass,
            diffusivity,
            dx,
            left_value=left_value,
            right_value=right_value,
            left_flux=left_flux,
            right_flux=right_flux,
        )

    def membrane_face_flux(
        self,
        water_mass,
        diffusivity,
        proton_current_face,
        lambda_face,
        dx,
        left_value=None,
        right_value=None,
    ):
        diffusion_flux = _diffusive_face_flux(
            water_mass,
            diffusivity,
            dx,
            left_value=left_value,
            right_value=right_value,
        )
        proton_current_face = np.asarray(proton_current_face, dtype=float)
        lambda_face = np.asarray(lambda_face, dtype=float)
        if proton_current_face.shape != diffusion_flux.shape or lambda_face.shape != diffusion_flux.shape:
            raise ValueError("质子电流和 lambda_face 必须与面通量同为 N+1 长度")
        drag_flux = (
            self.electro_osmotic_drag(lambda_face)
            * self.molar_mass
            * proton_current_face
            / F
        )
        return diffusion_flux + drag_flux

    @staticmethod
    def total_water_rhs(face_flux, reaction_water_source, dx):
        """相变不改变总水，只在独立冰相方程和能量方程中出现。"""
        return -_face_divergence(face_flux, dx) + np.asarray(
            reaction_water_source,
            dtype=float,
        )

    def ice_source(self, liquid_mass, ice_mass, temperature, freezing_temperature=T_FREEZE):
        """S_i>0 表示液水冻结，S_i<0 表示冰融化，单位 kg/(m^3 s)。"""
        liquid_mass = np.maximum(np.asarray(liquid_mass, dtype=float), 0.0)
        ice_mass = np.maximum(np.asarray(ice_mass, dtype=float), 0.0)
        temperature = np.asarray(temperature, dtype=float)
        freezing_temperature = np.asarray(freezing_temperature, dtype=float)
        return np.where(
            temperature < freezing_temperature,
            self.k_freeze * liquid_mass,
            -self.k_melt * ice_mass,
        )

    @staticmethod
    def ice_rhs(ice_source):
        return np.asarray(ice_source, dtype=float)

    @staticmethod
    def condensation_rate(liquid_mass_new, liquid_mass_old, ice_source, dt):
        """无液水对流时 S_cond = d(ml)/dt + S_i；正值冷凝，负值蒸发。"""
        if dt <= 0.0:
            raise ValueError("dt 必须为正")
        return (
            (np.asarray(liquid_mass_new, dtype=float) - np.asarray(liquid_mass_old, dtype=float))
            / dt
            + np.asarray(ice_source, dtype=float)
        )


@dataclass
class ChargeConservation:
    """备注 1 的准稳态电流分配：i_s+i_m=j(t)。"""

    @staticmethod
    def current_distribution(applied_current, x):
        x = np.asarray(x, dtype=float)
        electron_current = np.zeros_like(x)
        proton_current = np.zeros_like(x)
        x_agdl = L_AGDL
        x_acl = x_agdl + L_ACL
        x_pem = x_acl + L_PEM
        x_ccl = x_pem + L_CCL
        mask = x < x_agdl
        electron_current[mask] = applied_current
        mask = (x >= x_agdl) & (x < x_acl)
        xi = (x[mask] - x_agdl) / L_ACL
        electron_current[mask] = applied_current * (1.0 - xi)
        proton_current[mask] = applied_current * xi
        mask = (x >= x_acl) & (x < x_pem)
        proton_current[mask] = applied_current
        mask = (x >= x_pem) & (x < x_ccl)
        xi = (x[mask] - x_pem) / L_CCL
        proton_current[mask] = applied_current * (1.0 - xi)
        electron_current[mask] = applied_current * xi
        mask = x >= x_ccl
        electron_current[mask] = applied_current
        return electron_current, proton_current

    @staticmethod
    def reaction_sources(domain, applied_current):
        return {
            "h2": domain.h2_source(applied_current),
            "o2": domain.o2_source(applied_current),
            "water": domain.water_source(applied_current),
        }

    @staticmethod
    def residual(electron_current, proton_current, applied_current):
        return (
            np.asarray(electron_current, dtype=float)
            + np.asarray(proton_current, dtype=float)
            - np.asarray(applied_current, dtype=float)
        )


@dataclass
class EnergyConservation:
    """热传导 + 电化学热 + 冷凝潜热 + 冻结潜热。"""

    latent_condensation: float = LATENT_CONDENSATION
    latent_freezing: float = LATENT_FREEZING
    thermal_neutral_voltage: float = E_TH

    @staticmethod
    def effective_properties(
        domain,
        gas_porosity,
        liquid_fraction,
        ice_fraction,
        gas_density,
        gas_heat_capacity,
        gas_conductivity,
    ):
        """以附件干层表观物性为基线，加入孔隙流体热容及液水/冰导热修正。"""
        gas_porosity = np.asarray(gas_porosity, dtype=float)
        liquid_fraction = np.asarray(liquid_fraction, dtype=float)
        ice_fraction = np.asarray(ice_fraction, dtype=float)
        gas_density = np.asarray(gas_density, dtype=float)
        gas_heat_capacity = np.asarray(gas_heat_capacity, dtype=float)
        gas_conductivity = np.asarray(gas_conductivity, dtype=float)
        rho_cp = (
            domain.material_density * domain.material_heat_capacity
            + gas_porosity * gas_density * gas_heat_capacity
            + liquid_fraction * THERMAL_LIQUID[0] * THERMAL_LIQUID[1]
            + ice_fraction * THERMAL_ICE[0] * THERMAL_ICE[1]
        )
        conductivity = (
            domain.material_conductivity
            + liquid_fraction * (THERMAL_LIQUID[2] - gas_conductivity)
            + ice_fraction * (THERMAL_ICE[2] - gas_conductivity)
        )
        return rho_cp, np.maximum(conductivity, EPS_FLOOR)

    @staticmethod
    def face_heat_flux(temperature, conductivity, dx, left_flux=0.0, right_flux=0.0):
        return _diffusive_face_flux(
            temperature,
            conductivity,
            dx,
            left_flux=left_flux,
            right_flux=right_flux,
        )

    @staticmethod
    def convection_boundary_flux(
        left_temperature,
        right_temperature,
        ambient_temperature,
        h=H_CONVECTION,
    ):
        """返回沿 +x 的左右边界热通量；左侧向外散热对应负号。"""
        left_flux = -h * (np.asarray(left_temperature) - np.asarray(ambient_temperature))
        right_flux = h * (np.asarray(right_temperature) - np.asarray(ambient_temperature))
        return left_flux, right_flux

    def electrochemical_heat(self, current_density, voltage, active_thickness=L_TOTAL):
        if active_thickness <= 0.0:
            raise ValueError("active_thickness 必须为正")
        return (
            np.asarray(current_density, dtype=float)
            * (self.thermal_neutral_voltage - np.asarray(voltage, dtype=float))
            / active_thickness
        )

    def phase_change_heat(self, ice_source, condensation_rate=0.0):
        return (
            self.latent_freezing * np.asarray(ice_source, dtype=float)
            + self.latent_condensation * np.asarray(condensation_rate, dtype=float)
        )

    @staticmethod
    def temperature_rhs(
        volumetric_heat_capacity,
        heat_flux_face,
        reaction_heat,
        phase_change_heat,
        auxiliary_heat,
        dx,
    ):
        return (
            -_face_divergence(heat_flux_face, dx)
            + np.asarray(reaction_heat, dtype=float)
            + np.asarray(phase_change_heat, dtype=float)
            + np.asarray(auxiliary_heat, dtype=float)
        ) / np.maximum(np.asarray(volumetric_heat_capacity, dtype=float), EPS_FLOOR)


@dataclass
class CellVoltage:
    """冰堵后的可逆电压、活化损失、欧姆损失和浓差损失。"""

    alpha: float = ALPHA
    j0_ref: float = J0_REF
    activation_energy: float = EA_ACTIVATION

    @staticmethod
    def reversible_voltage(temperature, p_h2, p_o2):
        temperature = float(np.asarray(temperature, dtype=float))
        p_h2 = max(float(p_h2), EPS_FLOOR)
        p_o2 = max(float(p_o2), EPS_FLOOR)
        return float(
            1.229
            - 8.5e-4 * (temperature - T_REF)
            + R * temperature / (2.0 * F)
            * np.log((p_h2 / P_REF) * np.sqrt(p_o2 / P_REF))
        )

    def exchange_current_density(self, temperature):
        temperature = float(temperature)
        return float(
            self.j0_ref
            * np.exp(
                -self.activation_energy / R
                * (1.0 / temperature - 1.0 / T_REF)
            )
        )

    def activation_loss(self, current_density, temperature, active_area_factor):
        j0_effective = self.exchange_current_density(temperature) * max(
            float(active_area_factor),
            EPS_FLOOR,
        )
        return float(
            R * float(temperature) / (self.alpha * F)
            * np.arcsinh(float(current_density) / (2.0 * j0_effective))
        )

    @staticmethod
    def membrane_conductivity(temperature, lambda_water):
        temperature = np.asarray(temperature, dtype=float)
        lambda_water = np.asarray(lambda_water, dtype=float)
        conductivity = (0.5139 * lambda_water - 0.326) * np.exp(
            1268.0 * (1.0 / 303.15 - 1.0 / temperature)
        )
        return np.maximum(conductivity, EPS_FLOOR)

    def ohmic_loss(self, current_density, temperature_pem, lambda_pem, dx_pem):
        conductivity = self.membrane_conductivity(temperature_pem, lambda_pem)
        widths = _cell_widths(dx_pem, conductivity.size)
        membrane_asr = float(np.sum(widths / conductivity))
        total_asr = membrane_asr + R_CONTACT_ASR
        return float(current_density) * total_asr, membrane_asr, total_asr

    @staticmethod
    def limiting_current(diffusivity_o2, concentration_o2_ccl, dx_cathode):
        diffusivity_o2 = np.asarray(diffusivity_o2, dtype=float)
        widths = _cell_widths(dx_cathode, diffusivity_o2.size)
        transport_resistance = float(
            np.sum(widths / np.maximum(diffusivity_o2, EPS_FLOOR))
        )
        concentration = max(float(np.mean(concentration_o2_ccl)), 0.0)
        return float(4.0 * F * concentration / max(transport_resistance, EPS_FLOOR))

    def concentration_loss(
        self,
        current_density,
        temperature,
        diffusivity_o2,
        concentration_o2_ccl,
        dx_cathode,
    ):
        limiting_current = max(
            self.limiting_current(diffusivity_o2, concentration_o2_ccl, dx_cathode),
            EPS_FLOOR,
        )
        ratio = float(current_density) / limiting_current
        ratio_safe = np.clip(ratio, 0.0, 1.0 - 1.0e-12)
        loss = -R * float(temperature) / (4.0 * F) * np.log1p(-ratio_safe)
        return float(loss), float(limiting_current), bool(ratio >= 1.0)

    def compute(
        self,
        current_density,
        temperature_cell,
        p_h2_acl,
        p_o2_ccl,
        temperature_pem,
        lambda_pem,
        diffusivity_o2_cathode,
        concentration_o2_ccl,
        active_area_factor,
        dx_pem,
        dx_cathode,
    ):
        reversible = self.reversible_voltage(temperature_cell, p_h2_acl, p_o2_ccl)
        active_factor = float(np.clip(active_area_factor, EPS_FLOOR, 1.0))
        activation = self.activation_loss(current_density, temperature_cell, active_factor)
        ohmic, membrane_asr, total_asr = self.ohmic_loss(
            current_density,
            temperature_pem,
            lambda_pem,
            dx_pem,
        )
        concentration, limiting_current, transport_limited = self.concentration_loss(
            current_density,
            temperature_cell,
            diffusivity_o2_cathode,
            concentration_o2_ccl,
            dx_cathode,
        )
        voltage = reversible - activation - ohmic - concentration
        return {
            "cell_voltage": float(voltage),
            "reversible_voltage": float(reversible),
            "activation_loss": float(activation),
            "ohmic_loss": float(ohmic),
            "concentration_loss": float(concentration),
            "limiting_current": float(limiting_current),
            "active_area_factor": float(active_factor),
            "membrane_asr": float(membrane_asr),
            "total_asr": float(total_asr),
            "transport_limited": bool(transport_limited),
        }


In [ ]:
# Cell 3：Domain 基类及五个物理域子类
# 设计原则：Domain 只保存静态材料属性、传输模式和区域源项；
# 动态水相分配、气相孔隙率、FVM 通量与守恒方程均由 Cell 2 / 后续求解器负责。

WATER_TRANSPORT_MODES = frozenset({"none", "porous_vapor", "membrane"})


@dataclass
class Domain:
    """一维层域的静态材料属性、变量开关及电化学源项。"""

    name: str
    thickness: float
    porosity: float
    material_density: float
    material_heat_capacity: float
    material_conductivity: float
    permeability: float = 0.0
    contact_angle_deg: Optional[float] = None
    water_transport_mode: str = "none"
    ionomer_fraction: float = 0.0
    transports_h2: bool = False
    transports_o2: bool = False
    conducts_electrons: bool = False
    conducts_protons: bool = False
    is_catalyst_layer: bool = False

    def __post_init__(self):
        if self.thickness <= 0.0:
            raise ValueError(f"{self.name}: thickness 必须为正")
        if not 0.0 <= self.porosity < 1.0:
            raise ValueError(f"{self.name}: porosity 必须位于 [0, 1)")
        if self.material_density <= 0.0:
            raise ValueError(f"{self.name}: material_density 必须为正")
        if self.material_heat_capacity <= 0.0:
            raise ValueError(f"{self.name}: material_heat_capacity 必须为正")
        if self.material_conductivity <= 0.0:
            raise ValueError(f"{self.name}: material_conductivity 必须为正")
        if self.permeability < 0.0:
            raise ValueError(f"{self.name}: permeability 不得为负")
        if self.water_transport_mode not in WATER_TRANSPORT_MODES:
            raise ValueError(
                f"{self.name}: water_transport_mode 必须属于 "
                f"{sorted(WATER_TRANSPORT_MODES)}"
            )
        if not 0.0 <= self.ionomer_fraction <= 1.0:
            raise ValueError(f"{self.name}: ionomer_fraction 必须位于 [0, 1]")
        if self.water_transport_mode == "porous_vapor" and self.porosity <= 0.0:
            raise ValueError(f"{self.name}: porous_vapor 模式要求正孔隙率")

    @property
    def transports_water(self):
        """兼容布尔判断；具体机制由 water_transport_mode 决定。"""
        return self.water_transport_mode != "none"

    @staticmethod
    def _zero_like(value):
        return np.zeros_like(np.asarray(value, dtype=float))

    def h2_source(self, current_density):
        return self._zero_like(current_density)

    def o2_source(self, current_density):
        return self._zero_like(current_density)

    def water_source(self, current_density):
        return self._zero_like(current_density)

    def volume_reaction_current(self, current_density):
        return self._zero_like(current_density)

    def active_area_factor(self, ice_volume_fraction=0.0):
        return np.ones_like(np.asarray(ice_volume_fraction, dtype=float))


class AGDL(Domain):
    def __init__(self):
        rho, cp, k = THERMAL_GDL
        super().__init__(
            name="aGDL",
            thickness=L_AGDL,
            porosity=EPS_AGDL,
            material_density=rho,
            material_heat_capacity=cp,
            material_conductivity=k,
            permeability=PERM_GDL,
            contact_angle_deg=CONTACT_ANGLE_GDL,
            water_transport_mode="porous_vapor",
            transports_h2=True,
            conducts_electrons=True,
        )


class ACL(Domain):
    def __init__(self):
        rho, cp, k = THERMAL_CL
        super().__init__(
            name="aCL",
            thickness=L_ACL,
            porosity=EPS_ACL,
            material_density=rho,
            material_heat_capacity=cp,
            material_conductivity=k,
            permeability=PERM_CL,
            contact_angle_deg=CONTACT_ANGLE_CL,
            water_transport_mode="porous_vapor",
            ionomer_fraction=CL_IONOMER_FRACTION,
            transports_h2=True,
            conducts_electrons=True,
            conducts_protons=True,
            is_catalyst_layer=True,
        )

    def h2_source(self, current_density):
        return -np.asarray(current_density, dtype=float) / (2.0 * F * self.thickness)

    def volume_reaction_current(self, current_density):
        return np.asarray(current_density, dtype=float) / self.thickness


class PEM(Domain):
    def __init__(self):
        rho, cp, k = THERMAL_PEM
        super().__init__(
            name="PEM",
            thickness=L_PEM,
            porosity=0.0,
            material_density=rho,
            material_heat_capacity=cp,
            material_conductivity=k,
            water_transport_mode="membrane",
            conducts_protons=True,
        )


class CCL(Domain):
    def __init__(self):
        rho, cp, k = THERMAL_CL
        super().__init__(
            name="cCL",
            thickness=L_CCL,
            porosity=EPS_CCL,
            material_density=rho,
            material_heat_capacity=cp,
            material_conductivity=k,
            permeability=PERM_CL,
            contact_angle_deg=CONTACT_ANGLE_CL,
            water_transport_mode="porous_vapor",
            ionomer_fraction=CL_IONOMER_FRACTION,
            transports_o2=True,
            conducts_electrons=True,
            conducts_protons=True,
            is_catalyst_layer=True,
        )
        self.ice_area_exponent = BETA_ICE_ACTIVE_AREA

    def o2_source(self, current_density):
        return -np.asarray(current_density, dtype=float) / (4.0 * F * self.thickness)

    def water_source(self, current_density):
        return (
            M_WATER
            * np.asarray(current_density, dtype=float)
            / (2.0 * F * self.thickness)
        )

    def volume_reaction_current(self, current_density):
        return np.asarray(current_density, dtype=float) / self.thickness

    def active_area_factor(self, ice_volume_fraction=0.0):
        """建模假设：冰占初始孔隙的比例按 beta 次幂削弱 CCL 有效反应面积。"""
        ice_volume_fraction = np.asarray(ice_volume_fraction, dtype=float)
        available_pore_ratio = np.clip(
            1.0 - ice_volume_fraction / self.porosity,
            0.0,
            1.0,
        )
        return available_pore_ratio**self.ice_area_exponent


class CGDL(Domain):
    def __init__(self):
        rho, cp, k = THERMAL_GDL
        super().__init__(
            name="cGDL",
            thickness=L_CGDL,
            porosity=EPS_CGDL,
            material_density=rho,
            material_heat_capacity=cp,
            material_conductivity=k,
            permeability=PERM_GDL,
            contact_angle_deg=CONTACT_ANGLE_GDL,
            water_transport_mode="porous_vapor",
            transports_o2=True,
            conducts_electrons=True,
        )


DOMAINS = (AGDL(), ACL(), PEM(), CCL(), CGDL())
DOMAIN_BY_NAME = {domain.name: domain for domain in DOMAINS}

if len(DOMAIN_BY_NAME) != len(DOMAINS):
    raise ValueError("Domain 名称必须唯一")
if not np.isclose(sum(domain.thickness for domain in DOMAINS), L_TOTAL):
    raise ValueError("五个 Domain 的总厚度与 L_TOTAL 不一致")
